# UT1.6b - Accediendo a Google Docs

Conectar pandas con Google Sheets es muy directo, ya que las hojas de cálculo tienen estructura tabular nativa. El método depende de si la hoja es pública o privada.

## Opción 1: Hoja pública o con enlace compartido (Sin API ni credenciales)

Si la hoja tiene configurado el acceso como "Cualquier persona con el enlace puede ver", puedes leerla directamente exportándola como CSV usando su ID (el código largo que aparece en la URL entre /d/ y /edit).

In [9]:
%reset -f
%run notebook_setup.py
%whos

Variable   Type      Data/Info
------------------------------
ipytest    module    <module 'ipytest' from '/<...>ges/ipytest/__init__.py'>
np         module    <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
pd         module    <module 'pandas' from '/h<...>ages/pandas/__init__.py'>
pytest     module    <module 'pytest' from '/h<...>ages/pytest/__init__.py'>


In [13]:
sheet_id = "1Kd2NMaGmkXM6fMUeHsdtBE-3oa5CwZBxxRPYcOw3UzE"  # TU_ID_DE_LA_HOJA
nombre_pestana = "Febrero"  # Opcional: especifica la pestaña exacta

# Método 1: Exportación directa (lee la primera pestaña por defecto)
url_csv = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
df = pd.read_csv(url_csv)

print("Primera hoja:\n", df.head())

# Método 2: Seleccionar una pestaña específica por nombre usando gviz
url_pestana = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={nombre_pestana}"
df_pestana = pd.read_csv(url_pestana)

print(f"Hoja {nombre_pestana}:\n",df_pestana.head())

Primera hoja:
    id_producto                producto  unidades precio_unitario   total
0            1    Portátil Profesional         2          1250,5    2501
1            2            Ratón Óptico        10           25,75   257,5
2            3           Cámara Web HD         5              45     225
3            4  Impresora Multifunción         1          350,25  350,25
Hoja Febrero:
    id_producto                producto  unidades precio_unitario   total
0            1    Portátil Profesional         4          1250,5    5002
1            2            Ratón Óptico         1           25,75   25,75
2            3           Cámara Web HD         2              45      90
3            4  Impresora Multifunción         1          350,25  350,25


## Opción 2: Hojas privadas y lectura/escritura (gspread)

Para acceder a hojas privadas o modificar datos desde Python, la forma estándar es combinar gspread con una cuenta de servicio de Google Cloud.

1. Instala gspread y el módulo de autenticación interactiva OAuth: `pip install gspread google-auth-oauthlib`.
2. Obtener el archivo de credenciales OAuth (solo la primera vez): 
   _Necesario para que Google reconozca tu script local._
   
   1. Entra en Google Cloud Console y crea un proyecto (o selecciona uno existente).
   2. Ve a APIs y servicios > Biblioteca, busca y habilita Google Sheets API y Google Drive API.
   3. Ve a APIs y servicios > Pantalla de consentimiento de OAuth, elige Interno, rellena el nombre de la app y tu correo, y en Usuarios de prueba añade tu propio correo de Educarex.
   4. Ve a APIs y servicios > Credenciales > Crear credenciales > ID de cliente de OAuth.
   5. Selecciona tipo de aplicación Aplicación de escritorio, créala y descarga el archivo JSON.

3. Guardar el JSON en la ruta esperada por gspread:
   Renombra el archivo descargado a `credentials.json` y guárdalo en la carpeta de configuración de gspread de tu usuario:
   + Windows: `C:\Users\<TU_USUARIO>\AppData\Roaming\gspread\credentials.json`
   + Linux: `~/.config/gspread/credentials.json` 
   _(Si prefieres dejar `credentials.json` en la misma carpeta que tu script, indícalo en el código del siguiente paso)_

4. Ejecutar la autenticación interactiva en Python:
   Ejecuta este código en tu celda. Se abrirá automáticamente una pestaña en tu navegador para que inicies sesión y des permisos:
   

In [21]:
import gspread
   
# Abre el navegador para iniciar sesión y guarda el token para futuros usos
# Si dejaste el archivo en la carpeta actual, usa: gc = gspread.oauth(credentials_filename="credentials.json")
gc = gspread.oauth()
   
# Abre tu hoja de cálculo y pásala a pandas
spreadsheet = gc.open("UT1_Hoja_privada")
# Abrir el documento por su URL o ID y seleccionar la pestaña
#spreadsheet = gc.open_by_key("TU_ID_DE_LA_HOJA")
worksheet = spreadsheet.worksheet("Mayo")  
# O spreadsheet.sheet1 para la primera

# --- LEER DATOS A PANDAS ---
df = pd.DataFrame(worksheet.get_all_records())
print(df.head())



   id_producto                producto  unidades  precio_unitario total
0            1    Portátil Profesional        20            12505      
1            2            Ratón Óptico       100             2575      
2            3           Cámara Web HD        50               45      
3            4  Impresora Multifunción        10            35025      


Tras iniciar sesión la primera vez, gspread guarda un archivo `authorized_user.json` automáticamente, por lo que las próximas veces que ejecutes el código entrará directo sin volver a pedirte confirmación en el navegador.

### Ahora modificaremos la columna precio total

In [ ]:
import gspread
import pandas as pd

gc = gspread.oauth()
spreadsheet = gc.open("UT1_Hoja_privada")
worksheet = spreadsheet.worksheet("Mayo")  
# O spreadsheet.sheet1 para la primera

# Leer datos actuales
df = pd.DataFrame(worksheet.get_all_records())
print(df)

# 1. Esto solo cambia los datos en tu PC (local):
# ----------------------------------------------
# Ejemplo A: Recalcular "total" multiplicando cantidad por precio
df["total"] = df["unidades"] * df["precio_unitario"]
print(df)

# Ejemplo B: Modificar "total" aplicando un incremento (ej. +21% IVA)
df["IVA"] = (df["total"] * 0.21).round(2)


# 2. OBLIGATORIO: Esto es lo que realmente lo sube a Google Sheets:
# -----------------------------------------------------------------
df_limpio = df.fillna("")
worksheet.update(
    range_name="A1",
    values=[df_limpio.columns.values.tolist()] + df_limpio.values.tolist()
)

   id_producto                producto  unidades  precio_unitario total
0            1    Portátil Profesional        20            12505      
1            2            Ratón Óptico       100             2575      
2            3           Cámara Web HD        50               45      
3            4  Impresora Multifunción        10            35025      
   id_producto                producto  unidades  precio_unitario   total
0            1    Portátil Profesional        20            12505  250100
1            2            Ratón Óptico       100             2575  257500
2            3           Cámara Web HD        50               45    2250
3            4  Impresora Multifunción        10            35025  350250


{'spreadsheetId': '1WGnI0npp6CRTectAtOnqvwVmwKYnx4IA6_tjfnidt5Q',
 'updatedRange': 'Mayo!A1:F5',
 'updatedRows': 5,
 'updatedColumns': 6,
 'updatedCells': 30}